# Home Depot Product Search Relevance


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# 1. Clean up any previous failed attempts
!rm -rf /content/attributes
!rm -rf /content/product_descriptions
!rm -rf /content/sample_submission
!rm -rf /content/test
!rm -rf /content/train
!rm -rf /content/home-depot

# 2. Create the destination directory
!mkdir -p /content/attributes
!mkdir -p /content/product_descriptions
!mkdir -p /content/sample_submission
!mkdir -p /content/test
!mkdir -p /content/train
!mkdir -p /content/home-depot

# 3. Copy the ZIP file from Drive to Colab VM (Fast)
!cp -r /content/drive/MyDrive/attributes.csv.zip /content/attributes.zip
!cp -r /content/drive/MyDrive/product_descriptions.csv.zip /content/product_descriptions.zip
!cp -r /content/drive/MyDrive/sample_submission.csv.zip /content/sample_submission.zip
!cp -r /content/drive/MyDrive/test.csv.zip /content/test.zip
!cp -r /content/drive/MyDrive/train.csv.zip /content/train.zip
!cp -r /content/drive/MyDrive/home-depot-product-search-relevance.zip /content/home-depot.zip

# 4. Unzip directly into the target folder (Fast)
!unzip -q /content/attributes.zip -d/content/attributes
!unzip -q /content/product_descriptions.zip -d/content/product_descriptions
!unzip -q /content/sample_submission.zip -d/content/sample_submission
!unzip -q /content/test.zip -d/content/test
!unzip -q /content/train.zip -d/content/train
!unzip -q /content/home-depot.zip -d/content/home-depot

In [ ]:
ls

In [ ]:
import pandas as pd
train = pd.read_csv("/content/train/train.csv", encoding='latin1')
test = pd.read_csv("/content/test/test.csv", encoding='latin1')
product_descriptions = pd.read_csv("/content/product_descriptions/product_descriptions.csv")
attributes = pd.read_csv("/content/attributes/attributes.csv")

train

In [ ]:
train.nunique()

In [ ]:
train["relevance"].value_counts()

In [ ]:
print(len(train))

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history, title):
    plt.figure()
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
!pip install deep-translator
!pip install googletrans==4.0.0-rc1

In [ ]:
import pandas as pd
from deep_translator import LibreTranslator
from tqdm import tqdm
import time
from googletrans import Translator

LANGUAGES = {
    "en": "English",
    "es": "Spanish",
    "fr": "French",
    "de": "German"
}

In [ ]:
translator = Translator()

In [ ]:
def safe_translate(text, target_lang):
    if not isinstance(text, str) or text.strip() == "":
        return text
    try:
        return translator.translate(text, dest=target_lang).text
    except Exception:
        return text

In [ ]:
def augment_row(row):
    augmented_rows = []

    for lang in LANGUAGES:
        new_row = row.copy()

        if lang != "en":
            new_row["search_term"] = safe_translate(row["search_term"], lang)
            new_row["product_title"] = safe_translate(row["product_title"], lang)

        augmented_rows.append(new_row)

    return augmented_rows

In [ ]:
from sklearn.model_selection import train_test_split

# Split original examples before augmentation or learning any text features.
assert train['id'].is_unique, "Original example IDs must be unique"
train_original, validation_original = train_test_split(
    train, test_size=0.2, random_state=42
)
train_original = train_original.copy()
validation_original = validation_original.copy()
assert set(train_original['id']).isdisjoint(validation_original['id'])

train_sample = train_original.sample(frac=0.04, random_state=42)

In [ ]:
augmented_data = []

for _, row in tqdm(train_sample.iterrows(), total=len(train_sample)):
    augmented_data.extend(augment_row(row))

In [ ]:
augmented_sample = (
    pd.DataFrame(augmented_data, columns=train_original.columns)
    if augmented_data else train_original.iloc[:0].copy()
)

In [ ]:
# augment_row copies the original ID, including when translation fails.
train_augmented = pd.concat(
    [train_original, augmented_sample], ignore_index=True
)
assert set(train_augmented['id']) == set(train_original['id'])
assert set(train_augmented['id']).isdisjoint(validation_original['id'])

In [ ]:
print("Original labeled examples:", len(train))
print("Training examples before augmentation:", len(train_original))
print("Untouched validation examples:", len(validation_original))
print("Sampled for augmentation:", len(train_sample))
print("Augmented rows:", len(augmented_sample))
print("Final training size:", len(train_augmented))

## 1a: Separate training and validation features

All models share the original 80/20 split. Only training examples are augmented.
Validation text receives the same preprocessing, but never contributes to learned
vocabularies, embeddings, or scaling statistics. Split membership is by example
ID; different examples for the same product may appear in both partitions.

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def prepare_text(frame):
    prepared = frame.merge(
        product_descriptions, on="product_uid", how="left",
        sort=False, validate="many_to_one"
    )
    for column in ['search_term', 'product_title', 'product_description']:
        prepared[column] = prepared[column].fillna('').astype(str)
    prepared['search_term'] = prepared['search_term'].str.lower()
    prepared['product_text'] = (
        prepared['product_title'].str.lower() + ' ' +
        prepared['product_description'].str.lower()
    )
    assert prepared['id'].tolist() == frame['id'].tolist()
    return prepared

train_prepared = prepare_text(train_augmented)
validation_prepared = prepare_text(validation_original)

# Learn the character vocabulary exclusively from augmented training text.
all_text = ''.join(train_prepared['search_term']) + ''.join(train_prepared['product_text'])
chars = sorted(set(all_text))
char2idx = {'<PAD>': 0, '<UNK>': 1}
for c in chars:
    char2idx[c] = len(char2idx)
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(char2idx)
print("Character vocabulary size:", vocab_size)

def encode_text(text, char2idx):
    return [char2idx.get(c, char2idx['<UNK>']) for c in text]

MAX_QUERY_LEN = 40
MAX_PRODUCT_LEN = 400

def encode_partition(frame):
    queries = frame['search_term'].apply(lambda text: encode_text(text, char2idx))
    products = frame['product_text'].apply(lambda text: encode_text(text, char2idx))
    return (
        pad_sequences(queries, maxlen=MAX_QUERY_LEN, padding='post', truncating='post'),
        pad_sequences(products, maxlen=MAX_PRODUCT_LEN, padding='post', truncating='post')
    )

X_query_train, X_product_train = encode_partition(train_prepared)
X_query_val, X_product_val = encode_partition(validation_prepared)
y_train = train_prepared['relevance'].to_numpy()
y_val = validation_prepared['relevance'].to_numpy()
assert len(X_query_train) == len(X_product_train) == len(y_train)
assert len(X_query_val) == len(X_product_val) == len(y_val)
print("Training shapes:", X_query_train.shape, X_product_train.shape, y_train.shape)
print("Validation shapes:", X_query_val.shape, X_product_val.shape, y_val.shape)

**1b**

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense,
    Concatenate, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dropout

def build_encoder(vocab_size, embed_dim=32, lstm_units=64):
    input_seq = Input(shape=(None,), name="char_input")

    x = Embedding(
        input_dim=vocab_size,
        output_dim=embed_dim,
        mask_zero=True
    )(input_seq)

    x = Dropout(0.2)(x)

    x = LSTM(lstm_units)(x)

    return Model(input_seq, x, name="shared_encoder")

In [ ]:
encoder = build_encoder(vocab_size)

query_input = Input(shape=(MAX_QUERY_LEN,), name="query_input")
product_input = Input(shape=(MAX_PRODUCT_LEN,), name="product_input")

query_vec = encoder(query_input)
product_vec = encoder(product_input)

# Comparison
abs_diff = Lambda(lambda x: tf.abs(x[0] - x[1]))([query_vec, product_vec])
merged = Concatenate()([query_vec, product_vec, abs_diff])

# Regression head
x = Dense(128, activation="relu")(merged)
x = Dense(64, activation="relu")(x)
output = Dense(1, activation="linear")(x)

In [ ]:
model1 = Model(
    inputs=[query_input, product_input],
    outputs=output
)

model1.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model1.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

history = model1.fit(
    [X_query_train, X_product_train],
    y_train,
    validation_data=([X_query_val, X_product_val], y_val),
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    epochs=10,
    batch_size=64
)

In [ ]:
plot_training_history(
    history,
    "Character-level Siamese LSTM Training"
)

**1c**

### Text Encoding with TF-IDF Vectorization
combine product_title and search_term into a single text feature.

Use the TF-IDF vectorizer's default tokenizer; fit only on training text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_product_info = train_prepared['product_title'] + ' ' + train_prepared['search_term']
val_product_info = validation_prepared['product_title'] + ' ' + validation_prepared['search_term']
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_train = tfidf_vectorizer.fit_transform(train_product_info)
tfidf_val = tfidf_vectorizer.transform(val_product_info)
print("TF-IDF shapes:", tfidf_train.shape, tfidf_val.shape)

Validation uses the training TF-IDF vocabulary and IDF weights. No additional split is performed.

In [ ]:
X_train, X_val = tfidf_train, tfidf_val
assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)

In [ ]:
from sklearn.linear_model import Ridge

baseline_model = Ridge(alpha=1.0)

baseline_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_train_pred = baseline_model.predict(X_train)
y_val_pred = baseline_model.predict(X_val)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)

print("Train RMSE:", train_rmse)
print("Val RMSE:", val_rmse)
print("Train MAE:", train_mae)
print("Val MAE:", val_mae)

**2a**

In [ ]:
import re
import nltk
from nltk.tokenize import RegexpTokenizer

# Download basic nltk resources
nltk.download('punkt')

# 1. Define a tokenizer that keeps words, numbers with units (90°), and special patterns (#SC)
# This regex looks for:
# - Words/Numbers mixed with specific symbols (like 90°)
# - Standard alphanumeric words
# - Special patterns starting with #
token_pattern = r'\w+(?:°|/)?\w*|#\w+'
tokenizer = RegexpTokenizer(token_pattern)

def tokenize_text(text):
    if not isinstance(text, str):
        return []
    # Convert to lower case as done in 1a
    text = text.lower()
    return tokenizer.tokenize(text)


# Tokenize each partition independently; train Word2Vec only on training tokens.
for frame in (train_prepared, validation_prepared):
    frame['search_term_tokens'] = frame['search_term'].apply(tokenize_text)
    frame['product_text_tokens'] = frame['product_text'].apply(tokenize_text)

print(train_prepared['search_term_tokens'].head())
print(train_prepared['product_text_tokens'].head())

**2b**

In [ ]:
!pip install gensim
from gensim.models import Word2Vec
import multiprocessing

# 1. Prepare the corpus for Word2Vec
# Validation tokens are excluded from the Word2Vec training corpus
corpus = train_prepared['search_term_tokens'].tolist() + train_prepared['product_text_tokens'].tolist()

print(f"Training Word2Vec on {len(corpus)} sentences...")

# 2. Train Word2Vec
# vector_size=100 is standard, window=5 looks at context, min_count=2 ignores very rare typos
w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=2,
    workers=multiprocessing.cpu_count()
)

print("Word2Vec training complete.")
print(f"Vocabulary size: {len(w2v_model.wv)}")

# Test the embedding
# Check if it captured semantic similarity (e.g., 'hammer' should be close to 'nail' or 'tool')
if 'hammer' in w2v_model.wv:
    print("\nMost similar to 'hammer':")
    print(w2v_model.wv.most_similar('hammer', topn=3))
else:
    print("'hammer' not in vocabulary.")

## 2c: Word-level Siamese LSTM

Use the training-only Word2Vec vocabulary to encode word tokens. Index 0 is
masked padding and index 1 represents unknown or rare words. Initialize the
shared embedding layer with the learned 100-dimensional Word2Vec vectors and
fine-tune it on training examples. Validation uses the same fixed word mapping.
Query and product sequences are capped at 40 and 400 words respectively.

In [ ]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dropout
from tensorflow.keras.models import Model

# Use only words retained by Word2Vec, which was fitted on training tokens.
word2idx = {word: index + 2 for index, word in enumerate(w2v_model.wv.index_to_key)}
WORD_PAD_ID = 0
WORD_UNK_ID = 1
MAX_QUERY_WORDS = 40
MAX_PRODUCT_WORDS = 400
word_embedding_matrix = np.zeros(
    (len(word2idx) + 2, w2v_model.wv.vector_size), dtype=np.float32
)
# Unknown words get a deterministic initial vector and can learn during training.
word_embedding_matrix[WORD_UNK_ID] = np.random.default_rng(42).normal(
    0, 0.05, size=w2v_model.wv.vector_size
)
for word, index in word2idx.items():
    word_embedding_matrix[index] = w2v_model.wv[word]

def encode_word_partition(frame):
    def encode_column(column, maxlen):
        sequences = frame[column].apply(
            lambda tokens: [word2idx.get(word, WORD_UNK_ID) for word in tokens]
        )
        return pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
    return (
        encode_column('search_term_tokens', MAX_QUERY_WORDS),
        encode_column('product_text_tokens', MAX_PRODUCT_WORDS)
    )

X_word_query_train, X_word_product_train = encode_word_partition(train_prepared)
X_word_query_val, X_word_product_val = encode_word_partition(validation_prepared)
assert len(X_word_query_train) == len(X_word_product_train) == len(y_train)
assert len(X_word_query_val) == len(X_word_product_val) == len(y_val)

def build_word_encoder(embedding_matrix, lstm_units=64):
    input_seq = Input(shape=(None,), dtype='int32', name='word_input')
    x = Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        mask_zero=True,
        trainable=True,
        name='word2vec_embedding'
    )(input_seq)
    x = Dropout(0.2)(x)
    x = LSTM(lstm_units)(x)
    return Model(input_seq, x, name='shared_word_encoder')

In [ ]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate, Lambda, Dropout
from tensorflow.keras.models import Model

word_encoder = build_word_encoder(word_embedding_matrix)

query_input = Input(shape=(MAX_QUERY_WORDS,), dtype="int32", name="word_query_input")
product_input = Input(shape=(MAX_PRODUCT_WORDS,), dtype="int32", name="word_product_input")

query_vec = word_encoder(query_input)
product_vec = word_encoder(product_input)

# Comparison
abs_diff = Lambda(lambda x: tf.abs(x[0] - x[1]))([query_vec, product_vec])
merged = Concatenate()([query_vec, product_vec, abs_diff])

# Regression head
x = Dense(128, activation="relu")(merged)
x = Dense(64, activation="relu")(x)
output = Dense(1, activation="linear")(x)

In [ ]:
model2 = Model(
    inputs=[query_input, product_input],
    outputs=output
)

model2.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model2.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

history = model2.fit(
    [X_word_query_train, X_word_product_train],
    y_train,
    validation_data=([X_word_query_val, X_word_product_val], y_val),
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    epochs=10,
    batch_size=64
)

In [ ]:
plot_training_history(
    history,
    "Word-level Siamese LSTM Training"
)

**2d**

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def sentence_features(frame):
    queries = frame['search_term'].astype(str).tolist()
    products = (frame['product_title'] + " " + frame['product_description']).astype(str).tolist()
    query_embeddings = sbert_model.encode(
        queries, batch_size=32, show_progress_bar=True, convert_to_numpy=True
    )
    product_embeddings = sbert_model.encode(
        products, batch_size=32, show_progress_bar=True, convert_to_numpy=True
    )
    return np.concatenate([
        query_embeddings, product_embeddings,
        np.abs(query_embeddings - product_embeddings),
        query_embeddings * product_embeddings
    ], axis=1)

# The pretrained encoder is fixed; encode the existing partitions separately.
sbert_train = sentence_features(train_prepared)
sbert_val = sentence_features(validation_prepared)

In [ ]:
X_train, X_val = sbert_train, sbert_val
assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
from copy import deepcopy
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

def fit_mlp_with_validation(regressor, X_train, y_train, X_val, y_val,
                            max_epochs=200, patience=10, tol=0.0001):
    """Train one epoch at a time without splitting augmented training rows."""
    if regressor.early_stopping:
        raise ValueError("Disable the MLP's internal validation split")
    best_score = -np.inf
    best_model = None
    epochs_without_improvement = 0
    validation_scores = []
    for epoch in range(max_epochs):
        regressor.partial_fit(X_train, y_train)
        score = r2_score(y_val, regressor.predict(X_val))
        if not np.isfinite(score):
            raise ValueError("Validation R² must be finite")
        validation_scores.append(score)
        if score > best_score + tol:
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        # Restore the best score, including improvements smaller than tolerance.
        if score > best_score:
            best_score = score
            best_model = deepcopy(regressor)
        if epochs_without_improvement >= patience:
            break
    if best_model is None:
        raise ValueError("At least one training epoch is required")
    return best_model, validation_scores

regressor = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.1,
    batch_size=64,
    early_stopping=False,
    max_iter=200,
    random_state=42
)
regressor, validation_scores = fit_mlp_with_validation(
    regressor, X_train, y_train, X_val, y_val
)
print("MLP epochs run:", len(validation_scores))
print("Best validation R²:", max(validation_scores))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_train_pred = regressor.predict(X_train)
y_val_pred = regressor.predict(X_val)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)

print("Sentence-BERT Feature Extractor Results")
print("--------------------------------------")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Val RMSE:   {val_rmse:.4f}")
print(f"Train MAE:  {train_mae:.4f}")
print(f"Val MAE:    {val_mae:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Add small noise to true labels to separate points visually
jitter = np.random.normal(0, 0.02, size=len(y_val))

plt.figure(figsize=(6, 6))
plt.scatter(
    y_val + jitter,
    y_val_pred,
    alpha=0.25,
    s=10
)

plt.xlabel("True Relevance")
plt.ylabel("Predicted Relevance")
plt.title("Sentence-BERT: True vs Predicted Relevance")
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

df_plot = pd.DataFrame({
    "True Relevance": y_val,
    "Predicted Relevance": y_val_pred
})

plt.figure(figsize=(7, 5))
df_plot.boxplot(
    column="Predicted Relevance",
    by="True Relevance",
    grid=True
)

plt.title("Sentence-BERT Predictions by True Relevance")
plt.suptitle("")
plt.xlabel("True Relevance")
plt.ylabel("Predicted Relevance")
plt.show()